# 01 - Data Collection
## Download de precos historicos das acoes do S&P 500
Periodo: 2010-01-01 ate hoje.
Fonte: Yahoo Finance via yfinance + lista de tickers da Wikipedia.

In [1]:
import pandas as pd
import yfinance as yf
import datetime as dt
import warnings
warnings.filterwarnings("ignore")

### 1.1 Obter lista de tickers do S&P 500
Extraimos a tabela de componentes do S&P 500 direto da Wikipedia.
Alguns tickers usam ponto (ex: BRK.B), mas o Yahoo Finance espera hifen (BRK-B).

Motivação: A gente precisa da lista de quais empresas fazem parte do S&P 500. O S&P 500 nao e fixo, empresas entram e saem ao longo do tempo. Precisamos dos tickers (codigos de negociacao, tipo AAPL pra Apple, MSFT pra Microsoft) pra saber quais acoes baixar do Yahoo Finance.

 A Wikipedia só é a fonte mais prática porque tem uma tabela estruturada e publica com os 503 tickers atuais (503 porque algumas empresas tem duas classes de acoes). Outras opções seriam o site oficial da S&P Global (que exige cadastro), APIs pagas como Bloomberg ou Refinitiv, ou ate manter uma lista hardcoded num CSV.

Um ponto importante: a gente esta pegando a composicao **atual** do S&P 500 (abril 2026), mas baixando precos desde 2010. Isso gera um vies chamado **survivorship bias**. Empresas que faliram ou foram removidas do indice entre 2010 e hoje nao aparecem na lista atual, entao a gente so treina com as "sobreviventes", o que inflaciona artificialmente os resultados. Corrigir isso de verdade exigiria uma base com o historico de composicao do indice, que é dado pago. 

In [2]:
import requests
from io import StringIO

url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
headers = {"User-Agent": "Mozilla/5.0"}
response = requests.get(url, headers=headers)

sp500_table = pd.read_html(StringIO(response.text))[0]

tickers = sp500_table["Symbol"].str.replace(".", "-", regex=False).tolist()

print(f"Total de tickers: {len(tickers)}")
sp500_table[["Symbol", "Security", "GICS Sector"]].head(10)

Total de tickers: 503


,Symbol,Security,GICS Sector
0,MMM,3M,Industrials
1,AOS,A. O. Smith,Industrials
2,ABT,Abbott Laboratories,Health Care
3,ABBV,AbbVie,Health Care
4,ACN,Accenture,Information Technology
5,ADBE,Adobe Inc.,Information Technology
6,AMD,Advanced Micro Devices,Information Technology
7,AES,AES Corporation,Utilities
8,AFL,Aflac,Financials
9,A,Agilent Technologies,Health Care


### 1.2 Download de precos historicos via yfinance
Baixamos OHLCV (Open, High, Low, Close, Adj Close, Volume) diario de 2010 ate hoje.


In [3]:
start_date = "2010-01-01"
end_date = dt.date.today().strftime("%Y-%m-%d")

df = yf.download(
    tickers=tickers,
    start=start_date,
    end=end_date,
    group_by="ticker",
    auto_adjust=False,
    threads=True
)

print(f"Shape: {df.shape}")
print(f"Periodo: {df.index.min()} ate {df.index.max()}")
df.head()

[*********************100%***********************]  503 of 503 completed


Shape: (4102, 3018)
Periodo: 2010-01-04 00:00:00 ate 2026-04-24 00:00:00


Ticker     VLTO                                        KMB             \
Price      Open High Low Close Adj Close Volume       Open       High   
Date                                                                    
2010-01-04  NaN  NaN NaN   NaN       NaN    NaN  61.639500  61.860020   
2010-01-05  NaN  NaN NaN   NaN       NaN    NaN  61.658676  61.955894   
2010-01-06  NaN  NaN NaN   NaN       NaN    NaN  61.697029  61.697029   
2010-01-07  NaN  NaN NaN   NaN       NaN    NaN  61.054649  61.054649   
2010-01-08  NaN  NaN NaN   NaN       NaN    NaN  60.278046  60.364334   

Ticker                            ...       CSX                                \
Price             Low      Close  ...       Low     Close Adj Close    Volume   
Date                              ...                                           
2010-01-04  60.968361  61.620327  ...  5.363333  5.428889  4.089188  25173000   
2010-01-05  61.035477  61.668266  ...  5.423333  5.572222  4.197149  35913600   
2010-01-06  60.862896  60.910835  ...  5.535556  5.590000  4.210540  29757600   
2010-01-07  60.038349  60.508148  ...  5.511111  5.551111  4.181247  23382000   
2010-01-08  59.242569  60.124641  ...  5.515556  5.820000  4.383782  54567000   

Ticker            CAG                                                       
Price            Open       High        Low      Close  Adj Close   Volume  
Date                                                                        
2010-01-04  18.093386  18.124514  17.898832  17.984436  10.194066  4009714  
2010-01-05  17.992218  18.326847  17.859922  18.062258  10.238178  5517148  
2010-01-06  18.085604  18.171206  17.937742  18.000000  10.202887  4248724  
2010-01-07  18.038912  18.108950  17.906614  18.054476  10.233767  3484920  
2010-01-08  18.031128  18.031128  17.875486  17.914396  10.154365  2861695  

[5 rows x 3018 columns]

### 1.3 Reestruturar o DataFrame
O yfinance retorna colunas em MultiIndex (ticker, campo).
Vamos empilhar para formato long: indice = (date, ticker), colunas = OHLCV.

Precisamos ates entender e validar o formato atual

In [4]:
print(f"Tipo das colunas: {type(df.columns)}")
print(f"Levels: {df.columns.nlevels}")
print(f"Primeiras 20 colunas: {df.columns.tolist()[:20]}")
print(f"Shape: {df.shape}")

Tipo das colunas: <class 'pandas.MultiIndex'>
Levels: 2
Primeiras 20 colunas: [('VLTO', 'Open'), ('VLTO', 'High'), ('VLTO', 'Low'), ('VLTO', 'Close'), ('VLTO', 'Adj Close'), ('VLTO', 'Volume'), ('KMB', 'Open'), ('KMB', 'High'), ('KMB', 'Low'), ('KMB', 'Close'), ('KMB', 'Adj Close'), ('KMB', 'Volume'), ('ADSK', 'Open'), ('ADSK', 'High'), ('ADSK', 'Low'), ('ADSK', 'Close'), ('ADSK', 'Adj Close'), ('ADSK', 'Volume'), ('CBOE', 'Open'), ('CBOE', 'High')]
Shape: (4102, 3018)


In [5]:
df.columns = pd.MultiIndex.from_tuples(
    [(t.lower(), p.lower().replace(" ", "_")) for t, p in df.columns]
)

df = df.stack(level=0, future_stack=True)
df.index.names = ["date", "ticker"]

df = df[["open", "high", "low", "close", "adj_close", "volume"]]
df = df.dropna(subset=["close"])
df = df.sort_index()

print(f"Shape final: {df.shape}")
print(f"Tickers unicos: {df.index.get_level_values('ticker').nunique()}")
df.head(10)

Shape final: (1943062, 6)
Tickers unicos: 503


open       high        low      close  adj_close  \
date       ticker                                                          
2010-01-04 a       22.453505  22.625179  22.267525  22.389128  19.810986   
           aapl     7.622500   7.660714   7.585000   7.643214   6.412382   
           abt     26.000362  26.177889  25.870815  26.129908  18.207748   
           acgl     7.978889   8.022222   7.972222   7.994444   7.601905   
           acn     41.520000  42.200001  41.500000  42.070000  31.227369   
           adbe    36.650002  37.299999  36.650002  37.090000  37.090000   
           adi     31.790001  32.189999  31.610001  31.670000  21.829477   
           adm     31.480000  31.840000  31.330000  31.469999  20.289303   
           adp     38.226513  38.226513  37.489025  37.603161  25.521585   
           adsk    25.610001  25.830000  25.610001  25.670000  25.670000   

                        volume  
date       ticker               
2010-01-04 a         3815561.0  
           aapl    493729600.0  
           abt      10829095.0  
           acgl      4813200.0  
           acn       3650100.0  
           adbe      4710200.0  
           adi       2102700.0  
           adm       3472500.0  
           adp       3930120.0  
           adsk      2228600.0

### 1.4 Verificacao basica de qualidade

In [6]:
print("Valores nulos por coluna:")
print(df.isnull().sum())
print(f"\nLinhas duplicadas: {df.index.duplicated().sum()}")

records_per_ticker = df.groupby("ticker").size()
print(f"\nRegistros por ticker (min): {records_per_ticker.min()}")
print(f"Registros por ticker (max): {records_per_ticker.max()}")
print(f"Registros por ticker (mediana): {records_per_ticker.median():.0f}")

Valores nulos por coluna:
open         0
high         0
low          0
close        0
adj_close    0
volume       0
dtype: int64

Linhas duplicadas: 0

Registros por ticker (min): 124
Registros por ticker (max): 4102
Registros por ticker (mediana): 4102


Dados limpos. Zero nulos, zero duplicatas. A maioria dos tickers tem o historico completo (4102 dias desde 2010). Os que tem menos (minimo 124) sao empresas que entraram no S&P 500 recentemente, tipo Carvana ou AppLovin. Normal.

### 1.5 Salvar dados brutos

In [7]:
df.to_parquet("../data/raw/sp500_daily_prices.parquet")
print("Salvo em data/raw/sp500_daily_prices.parquet")
print(f"Tamanho: {df.shape[0]:,} linhas x {df.shape[1]} colunas")

Salvo em data/raw/sp500_daily_prices.parquet
Tamanho: 1,943,062 linhas x 6 colunas
